In [1]:
import os, sys
os.chdir('..')
sys.path.insert(0, os.getcwd())

from graph.pipeline import pipeline
from graph.state import AgentState
print("Importing pipeline done.")

/Users/burakegekaya/Desktop/AgentBench-TR/venv/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Importing pipeline done.


In [2]:
QUESTIONS = [
    "BERTurk ne zaman yayınlandı?",
    "AeroGuard sisteminin doğruluk oranı nedir?",
    "Attention mekanizması transformer mimarisinde nasıl çalışır?",
    "LangGraph state graph nasıl tanımlanır?",
    "Yapay zeka eğitiminde fine-tuning nedir?",
]
print(f"Loading {len(QUESTIONS)} test questions done.")

Loading 5 test questions done.


In [3]:
import os

from retrieval.bm25_index import load_all_chunks
chunks = load_all_chunks()
print("Chunk count:", len(chunks))

Loading chunks from AdvancedNetworkNotlarım_cleaned.txt...
  Generated 89 chunks.
Loading chunks from AeroGuard_README_cleaned.txt...
  Generated 3 chunks.
Loading chunks from AttentionIsAllYouNeed_cleaned.txt...
  Generated 2 chunks.
Loading chunks from BERTURK_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from IntroToAINotlarım_cleaned.txt...
  Generated 138 chunks.
Loading chunks from LANGCHAIN_README_cleaned.txt...
  Generated 2 chunks.
Loading chunks from LANGGRAPH_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from Multimodal_LLM_Paper_cleaned.txt...
  Generated 6 chunks.
Loading chunks from THQuAD_cleaned.txt...
  Generated 7 chunks.
Loading complete — 249 total chunks.
Chunk count: 249


In [4]:
results = []
for q in QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Running query: {q}")
    state = AgentState(query=q)
    result = pipeline.invoke(state)
    results.append(result)
    print(f"  final_answer    : {result['final_answer']}")
    print(f"  confidence_score: {result['confidence_score']}")
    print(f"  retry_count     : {result['retry_count']}")
    print(f"  agent_logs count: {len(result['agent_logs'])}")


Running query: BERTurk ne zaman yayınlandı?
Running PlannerAgent — generated 4 sub-tasks in 3185ms.
Loading chunks from AdvancedNetworkNotlarım_cleaned.txt...
  Generated 89 chunks.
Loading chunks from AeroGuard_README_cleaned.txt...
  Generated 3 chunks.
Loading chunks from AttentionIsAllYouNeed_cleaned.txt...
  Generated 2 chunks.
Loading chunks from BERTURK_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from IntroToAINotlarım_cleaned.txt...
  Generated 138 chunks.
Loading chunks from LANGCHAIN_README_cleaned.txt...
  Generated 2 chunks.
Loading chunks from LANGGRAPH_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from Multimodal_LLM_Paper_cleaned.txt...
  Generated 6 chunks.
Loading chunks from THQuAD_cleaned.txt...
  Generated 7 chunks.
Loading complete — 249 total chunks.
Building BM25 index...
Building complete — 249 chunks indexed.
Initializing ChromaDB collection at /Users/burakegekaya/Desktop/AgentBench-TR/storage/chroma...
Collection 'agentbench_tr' re

In [5]:
for i, result in enumerate(results):
    print(f"\n--- Question {i+1}: {QUESTIONS[i][:50]} ---")
    for log in result['agent_logs']:
        print(f"  [{log['agent']:12}] latency={log['latency_ms']:8.1f}ms | output: {str(log['output'])[:80]}")


--- Question 1: BERTurk ne zaman yayınlandı? ---
  [planner     ] latency=  3184.8ms | output: ['BERTurk hakkında bilgi topla.', "BERTurk'un yayın tarihini araştır.", 'Yayın t
  [search      ] latency=  8353.0ms | output: 19 unique chunks retrieved
  [validator   ] latency=  3815.1ms | output: 1 claims, 0 hallucination flags
  [synthesizer ] latency=   925.2ms | output: BERTurk 25.03.2020 tarihinde yayınlandı.

--- Question 2: AeroGuard sisteminin doğruluk oranı nedir? ---
  [planner     ] latency=  3536.9ms | output: ['AeroGuard sisteminin tanımını araştır.', 'AeroGuard sisteminin doğruluk oranı 
  [search      ] latency=   286.8ms | output: 16 unique chunks retrieved
  [validator   ] latency=  5460.2ms | output: 1 claims, 1 hallucination flags
  [search      ] latency=   271.3ms | output: 16 unique chunks retrieved
  [validator   ] latency=  4540.8ms | output: 1 claims, 1 hallucination flags
  [search      ] latency=   282.0ms | output: 16 unique chunks retrieved
  [validator   ] la

In [6]:
errors = [(QUESTIONS[i], r['error']) for i, r in enumerate(results) if r.get('error')]
if errors:
    print("Detecting errors:")
    for q, e in errors:
        print(f"  {q}: {e}")
else:
    print("Detecting errors — none found, all queries completed successfully.")

Detecting errors — none found, all queries completed successfully.
